<a href="https://colab.research.google.com/github/spirosChv/neuro208-tutorials/blob/main/7_NeuroAI/Notebook2_lif_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUBA-LIF Neuron: From Currents to Spikes

This notebook explores the behavior of a **Current-Based Leaky Integrate-and-Fire (CUBA-LIF)** neuron. We will:

1. **Constant current injection** — observe how the membrane potential integrates a fixed current and produces periodic spiking
2. **Synaptic input** — drive the neuron with a regular train of input spikes and observe the synaptic current dynamics
3. **F-I curve** — measure the neuron's firing rate as a function of injected current strength

### The CUBA-LIF equations

The neuron has two state variables updated at each timestep:

**Synaptic current** (driven by incoming spikes $s_t$):
$$I_{syn}[t] = \alpha_{syn} \cdot I_{syn}[t-1] + s_t$$

**Membrane potential** (driven by total current):
$$V_{mem}[t] = \alpha_{mem} \cdot V_{mem}[t-1] + R \cdot (1 - \alpha_{mem}) \cdot I_{total}[t]$$

where $\alpha_{mem} = e^{-dt/\tau_{mem}}$, $\alpha_{syn} = e^{-dt/\tau_{syn}}$, and $I_{total}$ includes both the synaptic current and any externally injected current.

When $V_{mem}$ crosses a **threshold**, the neuron emits a spike and the membrane resets.

In [ ]:
!pip install mplcyberpunk -q

In [ ]:
import torch
import matplotlib.pyplot as plt
import mplcyberpunk

## Neuron Model

In [ ]:
class CUBAPointLeaky:
    """
    Current-Based Leaky Integrate-and-Fire (CUBA-LIF) neuron.
    Spikes induce a decaying synaptic current, which then drives the membrane potential.
    """
    def __init__(self, threshold, reset, tau_mem, tau_syn, R, dt):
        self.threshold = threshold
        self.reset = reset

        # Precompute decay factors
        self.alpha_mem = torch.exp(torch.tensor(-dt / tau_mem))
        self.alpha_syn = torch.exp(torch.tensor(-dt / tau_syn))

        # Voltage scaling: R * (1 - alpha_mem)
        self.voltage_scale = R * (1.0 - self.alpha_mem)

    def step(self, spike_input, mem, isyn, external_current=0.0):
        """
        Advance the neuron by one timestep.

        Args:
            spike_input: incoming spike (0 or 1)
            mem: current membrane potential
            isyn: current synaptic current
            external_current: optional constant current injection

        Returns:
            spike, new_mem, new_isyn
        """
        # Update synaptic current (decays + incoming spike)
        new_isyn = self.alpha_syn * isyn + spike_input

        # Total current = synaptic + external
        total_current = new_isyn + external_current

        # Update membrane potential
        new_mem = self.alpha_mem * mem + self.voltage_scale * total_current

        # Spike generation
        spike = (new_mem > self.threshold).float()

        # Reset where spiked
        new_mem = new_mem * (1 - spike) + self.reset * spike

        return spike, new_mem, new_isyn

In [ ]:
# Default parameters (from the training pipeline)
DT       = 1e-4    # 0.1 ms timestep
TAU_MEM  = 10e-3   # 10 ms membrane time constant
TAU_SYN  = 2e-3    # 2 ms synaptic time constant
R        = 5       # membrane resistance
THRESHOLD = 1.0
RESET     = 0.0

SIM_TIME  = 100e-3  # 100 ms simulation
NUM_STEPS = int(SIM_TIME / DT)
time_ms = torch.arange(NUM_STEPS) * DT * 1000  # time axis in ms

neuron = CUBAPointLeaky(
    threshold=THRESHOLD, reset=RESET,
    tau_mem=TAU_MEM, tau_syn=TAU_SYN,
    R=R, dt=DT)

print(f"DT = {DT*1000:.2f} ms  \u2192  {NUM_STEPS} steps for {SIM_TIME*1000:.0f} ms")
print(f"alpha_mem = {neuron.alpha_mem:.4f}  (membrane decay per step)")
print(f"alpha_syn = {neuron.alpha_syn:.4f}  (synaptic decay per step)")

## Helper: simulate and record

In [ ]:
def simulate(neuron, num_steps, spike_inputs=None, external_current=0.0):
    """
    Run the neuron for num_steps and record all state variables.

    Args:
        neuron: CUBAPointLeaky instance
        num_steps: number of simulation steps
        spike_inputs: tensor of shape (num_steps,) with input spikes, or None
        external_current: constant current injection (scalar)

    Returns:
        dict with keys: 'mem', 'isyn', 'spikes' (each a list of length num_steps)
    """
    mem = torch.tensor(0.0)
    isyn = torch.tensor(0.0)

    rec = {'mem': [], 'isyn': [], 'spikes': []}

    for t in range(num_steps):
        s_in = spike_inputs[t] if spike_inputs is not None else torch.tensor(0.0)
        spike, mem, isyn = neuron.step(s_in, mem, isyn, external_current)
        rec['mem'].append(mem.item())
        rec['isyn'].append(isyn.item())
        rec['spikes'].append(spike.item())

    return rec

## 1. Response to Constant Current Injection

We bypass the synapse entirely and inject a fixed current directly into the membrane at every timestep. Below threshold current, the membrane charges to a steady state but never spikes. Above threshold, the neuron fires periodically — stronger current means higher firing rate.

In [ ]:
currents = [0.15, 0.22, 0.35, 0.6]

with plt.style.context("cyberpunk"):
    fig, axes = plt.subplots(len(currents), 1, figsize=(10, 2.5 * len(currents)), sharex=True)

    for ax, I_ext in zip(axes, currents):
        rec = simulate(neuron, NUM_STEPS, external_current=I_ext)
        n_spikes = int(sum(rec['spikes']))

        ax.plot(time_ms, rec['mem'], color='#08F7FE', linewidth=1.2)
        ax.axhline(THRESHOLD, color='gray', linestyle='--', linewidth=0.8, label='Threshold')
        ax.set_ylabel('$V_{mem}$', color='white')
        ax.set_title(f'$I_{{ext}}$ = {I_ext}  |  {n_spikes} spikes', fontsize=11, color='white')
        ax.set_ylim(-0.05, THRESHOLD + 0.15)
        ax.tick_params(colors='white')
        ax.legend(loc='upper right', fontsize=8)

    axes[-1].set_xlabel('Time (ms)', color='white')
    fig.tight_layout()
    plt.show()

## 2. Response to Synaptic Input (Regular Spike Train)

Now we drive the neuron through its **synapse**: we send in a regular train of input spikes with a fixed inter-spike interval (ISI). Each input spike causes a jump in the synaptic current $I_{syn}$, which then decays exponentially with time constant $\tau_{syn}$. The synaptic current in turn charges the membrane potential.

This two-stage filtering (spike → synaptic current → membrane potential) is what makes the CUBA model more biologically realistic than a simple LIF.

In [ ]:
ISIs_ms = [5, 10, 20]  # inter-spike intervals in milliseconds

with plt.style.context("cyberpunk"):
    fig, axes = plt.subplots(len(ISIs_ms), 3, figsize=(14, 3 * len(ISIs_ms)), sharex=True)

    for row, isi_ms in enumerate(ISIs_ms):
        # Convert ISI from ms to simulation steps
        isi_steps = int(isi_ms / (DT * 1000))

        # Create regular input spike train
        spike_inputs = torch.zeros(NUM_STEPS)
        spike_inputs[torch.arange(0, NUM_STEPS, isi_steps)] = 1.0

        rec = simulate(neuron, NUM_STEPS, spike_inputs=spike_inputs)
        n_out = int(sum(rec['spikes']))

        # Input spikes
        ax0 = axes[row, 0]
        spike_times = time_ms[spike_inputs > 0].numpy()
        ax0.eventplot(spike_times, lineoffsets=0.5, linelengths=0.8, color='#FFD300')
        ax0.set_ylabel('Input spike', color='white')
        ax0.set_title(f'Input (ISI = {isi_ms} ms)', fontsize=11, color='white')
        ax0.set_ylim(0, 1)
        ax0.set_yticks([])
        ax0.tick_params(colors='white')

        # Synaptic current
        ax1 = axes[row, 1]
        ax1.plot(time_ms, rec['isyn'], color='#00ff41', linewidth=1.2)
        ax1.set_ylabel('$I_{syn}$', color='white')
        ax1.set_title(f'Synaptic current ($\\tau_{{syn}}$ = {TAU_SYN*1000:.0f} ms)', fontsize=11, color='white')
        ax1.tick_params(colors='white')

        # Membrane potential
        ax2 = axes[row, 2]
        ax2.plot(time_ms, rec['mem'], color='#08F7FE', linewidth=1.2)
        ax2.axhline(THRESHOLD, color='gray', linestyle='--', linewidth=0.8)
        ax2.set_ylabel('$V_{mem}$', color='white')
        ax2.set_title(f'Membrane potential  |  {n_out} output spikes', fontsize=11, color='white')
        ax2.set_ylim(-0.05, THRESHOLD + 0.15)
        ax2.tick_params(colors='white')

    for ax in axes[-1]:
        ax.set_xlabel('Time (ms)', color='white')

    fig.tight_layout()
    plt.show()

## 3. F-I Curve

The **frequency-current (F-I) curve** characterizes how a neuron's firing rate depends on the strength of a constant injected current. Below a critical current (the **rheobase**), the neuron never spikes. Above it, the firing rate increases monotonically.

In [ ]:
current_range = torch.linspace(0.0, 1.0, 200)
firing_rates = []

for I_ext in current_range:
    rec = simulate(neuron, NUM_STEPS, external_current=I_ext.item())
    n_spikes = sum(rec['spikes'])
    rate_hz = n_spikes / SIM_TIME  # convert to Hz
    firing_rates.append(rate_hz)

with plt.style.context("cyberpunk"):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(current_range.numpy(), firing_rates, color='#08F7FE', linewidth=2)
    ax.set_xlabel('Injected current $I_{ext}$', fontsize=12, color='white')
    ax.set_ylabel('Firing rate (Hz)', fontsize=12, color='white')
    ax.set_title('F-I Curve (CUBA-LIF)', fontsize=13, color='white')
    ax.tick_params(colors='white')
    mplcyberpunk.add_glow_effects(ax)

    # Mark the rheobase (first current that produces a spike)
    for i, rate in enumerate(firing_rates):
        if rate > 0:
            rheobase = current_range[i].item()
            ax.axvline(rheobase, color='#FE53BB', linestyle='--', linewidth=0.8, alpha=0.7)
            ax.annotate(f'Rheobase $\\approx$ {rheobase:.2f}',
                        xy=(rheobase, rate), xytext=(rheobase + 0.1, rate + 50),
                        arrowprops=dict(arrowstyle='->', color='#FE53BB'),
                        fontsize=10, color='#FE53BB')
            break

    fig.tight_layout()
    plt.show()

## Summary

The CUBA-LIF neuron exhibits the core behavior of biological spiking neurons:

- **Sub-threshold integration**: the membrane potential charges toward a steady state without spiking when input is weak
- **Threshold and reset**: once the membrane crosses the threshold, a spike is emitted and the potential resets
- **Synaptic filtering**: input spikes are first converted to a decaying synaptic current ($\tau_{syn}$) before driving the membrane ($\tau_{mem}$) — this two-stage filtering smooths out the input
- **Rate coding**: stronger inputs produce higher firing rates, as captured by the F-I curve

In the next notebook, we will use these neurons as building blocks for a **spiking neural network** trained with surrogate gradients.